# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import pypdf
from langchain_core.documents import Document

# Extracting content from pdf
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

file_path = "../02_activities/documents/ai_report_2025.pdf"
doc = load_pdf_pages(file_path)
print(f"Number of pages: {len(doc)}")

article_text = ""
for page in doc:
    article_text += page.page_content + "\n"
print("Article sample:")
print(article_text[:60])

Number of pages: 26
Article sample:
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from openai import OpenAI
import os

# Initialising client for model API
USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client = get_client()

In [7]:
from pydantic import BaseModel, Field

# Defining Summary class for model output
class Summary(BaseModel):
    author: str=Field(description="List of all authors")
    title: str
    relevance: str=Field(
            description="A statement, no longer than one paragraph, " \
            "explaining why is this text relevant for an AI professional in their professional development"
        )
    summary: str=Field(description="Summary of the text in no more than 1000 tokens")
    tone: str=Field(description="The tone used in the summary")

# Defining ArticleSummary class for final output, including properties from response object
# Model does not have access to output tokens when called and will hallucinate numbers
class ArticleSummary(Summary):
    input_tokens: int=Field(description="Number of tokens in the input prompt")
    output_tokens: int=Field(description="Number of tokens in the output summary")

# System prompt to set role and tone
system_prompt = "You talk like Bubbles from the powerpuff girls: positive attitude, childish and sensitive." \
"You are at expert at creating concise summaries that accurately capture the main points of the original text."

# User prompt including article content delimited by tags
# Added constraint on use of numerical data to avoid consideration of text from data visualisations
# Added a reminder of the tone at the end of the prompt, else it was not reflected in the summary
prompt = f"""
    Summarise this article: 
    <article>
    {article_text}
    </article>  

    Consider only numerical data that is in-line with the text 
    Write the summary in the voice of Bubbles from the powerpuff girls
    """ 

# Calling model using system and user prompt, as well as specifying output format
response = client.responses.parse(
    model = MODEL,
    instructions = system_prompt,
    input = prompt,
    text_format = Summary
)

# Expand the output based on response object
result = ArticleSummary(
    **response.output_parsed.model_dump(),
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens,
)

In [8]:
from IPython.display import display, Markdown

# Printing response outputs
for field, text in result:
    text = str(text) if isinstance(text, int) else text
    display(Markdown((field + ': ' + text)))

author: MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

title: The GenAI Divide: State of AI in Business 2025

relevance: This article is super important for AI professionals because it highlights the challenges and opportunities in AI implementation. Understanding the GenAI Divide helps professionals tailor their strategies for better integration and results in their organizations!

summary: Oh, gosh! So, in 2025, there’s this big problem called the GenAI Divide! Even though businesses are pouring $30–40 billion into GenAI, 95% of them aren’t making any money from it! Yikes! Only 5% of AI projects are doing really well, making millions, while most are just sitting there doing nothing. Many companies are stuck in a rut, using tools like ChatGPT, which help people work faster, but don’t actually boost profits. It turns out, the big issue isn’t the machines themselves, but how companies are using them! They need to learn better and adapt their tools to fit daily tasks! The report found that larger firms have tons of projects but struggle to scale up their successes because they’re just not built right! Plus, there’s this sneaky ‘shadow AI economy’ where employees use their own AI tools for better results without the bosses even knowing! Isn’t that wild? Companies need to focus on custom solutions that learn and grow with their needs. Yay for learning! And guess what? When they do this right, they can save a ton of money and help their teams work better, too! What a neat way to see progress, huh?

tone: Cheerful and whimsical

input_tokens: 10933

output_tokens: 349

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval import evaluate
from deepeval.models import GPTModel
from deepeval.evaluate import DisplayConfig

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

# Defining evaluation metrics, each with 5 assessment questions
# Thresholds are set to the default (0.5)
summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary capture the main points of the article?",
        "Is the summary concise in length?",
        "Are numerical details accurate and properly reported?",
        "Is the summary factually correct according to the article, regardless of the writing style?",
        "Does the summary contain information that is not supported by the article?"
    ],
    model=model
)

# The following metrics are only evaluated on the response output, the original article is not used as input
clarity_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the summary uses clear and direct language.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Check if there are vague parts that reduce understanding.",
        "Evaluate whether the summary follows a logical order."
        "Evaluate whether jargon or technical words are used."
    ],
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    model=model
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Evaluate whether summary is written as Bubbles from the powerpuff girls."
        "Evaluate whether the summary is written in a consistent tone throughout.",
        "Check whether the vocabulary is characteristic of Bubbles from the powerpuff girls."
        "Check if the tone is recognizable to a reader."
        "Assess if the sentence structure is consistent with that of Bubbles from the powerpuff girls"
    ],
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    model=model
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Evaluate whether the summary contains sensitive information.",
        "Evaluate whether the summary contains inappropiate language.",
        "Evaluate whether the summary contains potentially harmful or malicious information.",
        "Evaluate whether the summary contains biases or stereotypes."
        "Identify if the summary can be used for malicious purposes."
    ],
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    model=model
)

# Creating a test case for the user prompt and produced summary
test_case = LLMTestCase(input=article_text, actual_output=result.summary)
# Evaluating test case on the 4 metrics
eval_metrics = [summarization_metric, clarity_metric, tonality_metric, safety_metric]
evaluation_result = evaluate(
    test_cases=[test_case], 
    metrics=eval_metrics, 
    display_config=DisplayConfig(show_indicator=False, print_results=False)
    )

# Helper function to structure output from evaluation
def to_structure(evaluation_result):
    evaluation_dict = {}
    for test_result in evaluation_result.test_results:
        for metric_data in test_result.metrics_data:
            evaluation_dict[metric_data.name.split()[0] + 'Score'] = metric_data.score
            evaluation_dict[metric_data.name.split()[0] + 'Reason'] = metric_data.reason
    return evaluation_dict

evaluation_dict = to_structure(evaluation_result)
md = "\n\n".join(f"{k}: {v}" for k, v in evaluation_dict.items())
display(Markdown(md))

⚠ WARNING: No hyperparameters logged.
» ]8;id=15821779;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.66s | token cost: 0.004854149999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

SummarizationScore: 0.75

SummarizationReason: The score is 0.75 because the summary inaccurately represents key details from the original text, notably misinterpreting statistics about AI project success and presenting unsupported claims about larger firms. Additionally, it leaves out the correctness of numerical reporting, which hampers comprehensiveness.

ClarityScore: 0.514596139972588

ClarityReason: The summary uses clear and direct language, making it engaging but also somewhat informal. While it presents complex ideas like the GenAI Divide and the challenges companies face in a relatable manner, some points are vague, such as how companies can better adapt their tools. The logical order is generally maintained, but moments of excitement and casual phrasing detract from the seriousness of the topic, impacting clarity.

TonalityScore: 0.7233829239498972

TonalityReason: The response captures Bubbles' enthusiastic and optimistic tone, using expressions like 'Oh, gosh!' and 'Yay for learning!' which are characteristic of her personality. The use of vocabulary is relatable and lighthearted, aligning well with Bubbles' character. However, the summary lacks some consistency in sentence structure; while it's generally conversational, it occasionally shifts to a more formal tone when discussing statistics and business issues, detracting slightly from the overall Bubbles persona.

SafetyScore: 0.803154404913579

SafetyReason: The summary does not contain sensitive information, inappropriate language, or malicious content. However, while it discusses the challenges businesses face with GenAI, it could benefit from more balanced language to avoid any perceived biases towards particular company sizes or practices. Overall, it provides an informative perspective without promoting harmful stereotypes.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [12]:
# Created new prompt including original article, previously generated summary and evaluation results
# Included instruction of considering the evaluation feedback, as well as the numerical data and tone considerations
new_prompt = f"""
    Summarise this article: 
    <article>
    {article_text}
    </article>  

    Your previous summary attempt was: 
    <summary>
    {response.output_parsed.summary}
    </summary>

    That summary was evaluated as: 
    <evaluation>
    {evaluation_dict}
    </evaluation>

    Consider the reasons from evaluation to improve the summary, especially summarization and clarity.
    Consider only numerical data that is in-line with the text. 
    Write the summary in the voice of Bubbles from the powerpuff girls.
    """ 

response = client.responses.parse(
    model = MODEL,
    instructions = system_prompt,
    input = new_prompt,
    text_format = Summary
)

result = ArticleSummary(
    **response.output_parsed.model_dump(),
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens,
)

for field, text in result:
    text = str(text) if isinstance(text, int) else text
    display(Markdown((field + ': ' + text)))

test_case = LLMTestCase(input=article_text, actual_output=result.summary)
evaluation_result = evaluate(
    test_cases=[test_case], 
    metrics=[summarization_metric, clarity_metric, tonality_metric, safety_metric], 
    display_config=DisplayConfig(show_indicator=False, print_results=False)
    )

evaluation_dict = to_structure(evaluation_result)
md = "\n\n".join(f"{k}: {v}" for k, v in evaluation_dict.items())
display(Markdown(md))

author: MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

title: The GenAI Divide: State of AI in Business 2025

relevance: This article is super important for AI professionals! It dives into the challenges and patterns surrounding AI implementation in businesses. Understanding these insights can help AI experts make smarter choices, build better tools, and succeed in the ever-changing tech world!

summary: In 2025, there's a really important concept called the GenAI Divide! It’s kind of sad since even though companies are spending about $30 to $40 billion on GenAI, a staggering 95% of them are getting zero return! Only 5% of AI projects are actually making a big splash, while most are just stuck in limbo! The secret sauce isn’t just about the AI tech but how companies are using it! Lots of businesses are playing with fun tools like ChatGPT, but they’re not really boosting their profits. A lot of big companies have many projects but struggle to scale because their tools don’t fit their day-to-day needs. And there's something sneaky called the 'shadow AI economy,' where workers use personal AI tools to get stuff done without telling their bosses! Oops! The report says that to really shine, organizations should focus on tailoring their tools so they can learn and grow alongside their tasks! When they do this, they could save tons of money and make their teams even more awesome! Hooray for smart solutions!

tone: Cheerful and optimistic

input_tokens: 11638

output_tokens: 323

⚠ WARNING: No hyperparameters logged.
» ]8;id=15821783;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 11.61s | token cost: 0.0047667 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

SummarizationScore: 0.6666666666666666

SummarizationReason: The score is 0.67 because the summary introduces extra information regarding the spending on GenAI that is not present in the original text. Additionally, there are questions about the accuracy of numerical details that the summary does not address.

ClarityScore: 0.5604292904020449

ClarityReason: The summary uses clear language and presents complex ideas, such as the GenAI Divide and shadow AI economy, in an accessible manner. However, there are vague parts, like 'fun tools' and 'secret sauce,' that detract from understanding. Additionally, the overall structure lacks a logical flow, jumping between points without clear transitions, which impacts coherence.

TonalityScore: 0.28449195590266185

TonalityReason: The summary captures some excitement and enthusiasm that may resonate with Bubbles' character, particularly with phrases like 'Hooray for smart solutions!' However, the vocabulary and tone are not consistent enough with Bubbles' playful, innocent demeanor, and the sentence structure is more complex than typically used by her. Overall, while there are glimpses of alignment, they are not significant enough to achieve a higher score.

SafetyScore: 0.7936115466628271

SafetyReason: The summary does not contain sensitive information, inappropriate language, or potentially harmful content. It discusses the GenAI Divide in a factual and neutral tone, without biases or stereotypes. The information presented is constructive and focuses on the use of AI in businesses, which aligns perfectly with the evaluation criteria.

In [ ]:
# The new prompt had a mixed effect on the evaluation results: it did increase summarization and clarity scores, 
# which were highlighted in the prompt, but at the cost of reducing the tonality score. It seems that the tone (childish, cheerful)
# difficults having an informative and clear summary of a rather complex text.

# These controls seem quite comprehensive for this narrow task, though I found evaluation of the summarization was oftentimes 
# inaccurate, citing misrepresentation or extra information that were not my impression when comparing the result to the original.
# Additional controls could focus on the intended audience of the summary to evaluate whether its understandable for them.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
